In [ ]:
from langchain import OpenAI, PromptTemplate, LLMChain
from langchain.prompts import SystemMessagePromptTemplate,HumanMessagePromptTemplate,ChatPromptTemplate
from langchain.text_splitter import CharacterTextSplitter
from langchain.chains.mapreduce import MapReduceChain
from langchain.prompts import PromptTemplate
from langchain.chains.summarize import load_summarize_chain
from langchain.chat_models import ChatOpenAI
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chat_models import ChatOpenAI


In [ ]:
path='YOUR_INPUT_TRANSCRIPTION'
model_name="CHOSEN_OPENAI_MODEL"
api_key="YOUR_OPENAI_KEY"

In [ ]:
text_splitter = CharacterTextSplitter()
with open(path) as f:
    state_of_the_union = f.read()
texts = text_splitter.split_text(state_of_the_union)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=500)

In [ ]:
docs = text_splitter.create_documents([state_of_the_union])
llm = ChatOpenAI(max_tokens=5000, temperature=0, model_name=model_name,openai_api_key=api_key)

In [ ]:
prompt_template = """Please summarize the following text.

Write a summary below:
{text}

SUMMARY IN BRAZILIAN PORTUGUESE:"""

In [ ]:
system_message_prompt = SystemMessagePromptTemplate.from_template(prompt_template)
human_template="{text}"
human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)
prompt = ChatPromptTemplate.from_messages(
    [system_message_prompt, human_message_prompt],
)

In [ ]:
chain = load_summarize_chain(llm, chain_type="map_reduce", return_intermediate_steps=True, map_prompt=prompt, combine_prompt=prompt)
summary = chain({"input_documents": docs}, return_only_outputs=True)

In [ ]:
summary